### Import Dependencies

In [68]:
from google import genai
from google.genai import types
import os
from qdrant_client import QdrantClient
from langsmith import traceable, get_current_run_tree
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [69]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

### Embedding function

In [70]:
### Other task_type = SEMANTIC_SIMILARITY, CLASSIFICATION, CLUSTERING, RETRIEVAL_DOCUMENT, CODE_RETRIEVAL_QUERY, QUESTION_ANSWERING, FACT_VERIFICATION
@traceable(
    name="embed_query",
    run_type="embedding",
    metadata={
        "ls_provider": "google",
        "ls_model_name": "gemini-embedding-001"
    }
)
def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

### Retrieval Function

In [71]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [72]:
@traceable(
    name="retrieve_data",
    run_type="retriever"
)
def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [73]:
### Format retrieved context function
@traceable(
    name="format_retrieved_context",
    run_type="prompt"
)
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

### Create prompt template function

In [74]:
@traceable(
    name="build_prompt",
    run_type="prompt"
)
def build_prompt(query, preprocessed_context):
    prompt = f"""
    You are a helpful assistant that can answer questions about the products in stock.
    You are given a question and a list of context.
    You need to answer the question based on the product descriptions.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use the word "context" in your answer, refer it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question: 
    {query}
    """
    return prompt

### Generate Answer

In [75]:
@traceable(
    name="generate_answer",
    run_type="llm",
    metadata={
        "ls_provider": "google",
        "ls_model_name": "gemini-2.5-flash"
    }
)
def generate_answer(prompt):
    response = gemini_client.models.generate_content(
    contents=[prompt],
    model="gemini-2.5-flash",
)
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage_metadata.prompt_token_count,
            "output_tokens": response.usage_metadata.candidates_token_count,
            "total_tokens": response.usage_metadata.total_token_count,
        }
    return response.text

### Combine RAG Pipeline

In [76]:
@traceable(
    name="rag_pipeline"
)
def rag_pipeline(query, topk_k=5):
    retrieved_context = retrieve_data(query, topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(query, preprocessed_context)
    answer = generate_answer(prompt)
    return answer

In [77]:
print(rag_pipeline("do you have any smart watches?"))

Yes, we have several smart watches available:

- A Tykoit Smartwatch 41mm with heart rate monitor, sleep tracking, SpO2, and IP68 waterproof fitness tracking, compatible with Android and iOS phones. It features all-day activity tracking, customizable watch faces, smart notifications, and long-lasting battery life.
- A Military Smart Watch for men that allows answering and making calls, is 5 ATM waterproof, and includes a rugged fitness tracker with heart rate, SpO2, and AI Voice. It's designed for adventurers with military-grade tests and comprehensive health management.
- The PlayBetter Garmin Instinct 2 Solar Tactical (Black) Rugged GPS Smartwatch, which is a military-grade watch offering unlimited battery life in smartwatch mode (with solar charging), tactical features like stealth mode, GPS/GLONASS/Galileo, and all-day health monitoring including heart rate, sleep, and blood oxygen level. This comes as a power bundle with screen protectors and a portable charger.


In [78]:
print(rag_pipeline("could you suggest me some headphones? preferably with a rating of over 4 stars"))

Certainly, based on the available products, here are some headphones with a rating of over 4 stars:

- ID: B09V1F24X4, rating: 4.2, description: Wireless Earbuds with 4-Mic and Wireless Charging Case Waterproof 50H Playback Bluetooth Headphones LED Power Display Stereo Earphones, Touch Control in-Ear Headset with USB-C for Sports Work Black
- ID: B0BD8FXHPB, rating: 4.4, description: WeurGhy Wireless Earbuds, Bluetooth 5.3 Headphones, Bluetooth Earbuds with Stereo Sound, 42 Hrs of Playtime, IPX7 Waterproof, LED Display, Over Ear Earphones with Mic for Sports Running Workout Gym
- ID: B0BNNTDHNX, rating: 4.4, description: VuyKoo Bluetooth Headphones with Microphone/RGB LED Light Up, Cat Ear Wireless Headphones, Stereo Gaming Headset for Cellphone/PC/Laptop/Tablet/TV Kids Girls & Boys Teens/Birthday Gift (Purple)
- ID: B09XKJFXB4, rating: 4.4, description: Rhystereo Active Noise Cancelling Headphones, Black Wireless Headphones w/Mics, 45H Playtime USB-C Fast Charge, Deep Bass, BT 5.0, Co